# Clasificación de sonidos ambientales de ESC-50 con el Audio Spectrogram Transformer (AST)

**Miniproyecto 4 · Técnicas de Deep Learning (MAIA-4213) · Maestría en Inteligencia Artificial, Universidad de los Andes**

**Autores:** Miguel Muñoz, David Cardenas y Mario Celis

**Objetivo (enunciado):** hacer *fine-tuning* y evaluar el modelo preentrenado Audio Spectrogram Transformer (AST) para clasificar sonidos ambientales a partir de sus espectrogramas.

**Datos:** ESC-50 [1], cargado desde Hugging Face tal como indica el enunciado (`load_dataset("confit/esc50-demo", "fold1", ...)`). **Modelo:** `MIT/ast-finetuned-audioset-10-10-0.4593` [2].

**Sobre las categorías.** El enunciado menciona categorías como "aves", "agua", "viento" y "mamíferos", pero ESC-50 tiene **50 clases** (por ejemplo, *chirping birds*, *rain*, *wind*, *dog*) agrupadas en **5 categorías generales**: animales, paisajes naturales y agua, sonidos humanos no verbales, sonidos domésticos y sonidos urbanos. El modelo se entrena con las 50 clases, que es la tarea estándar del dataset, y los resultados se analizan también al nivel de las 5 categorías.

**Sobre la validación.** El código del enunciado llama `val_dataset` al split `test` del fold 1. Aquí ese conjunto se reserva como **prueba** y solo se usa para el reporte final. Las decisiones (estrategia de ajuste, aumento de datos) se toman con una **validación** separada del entrenamiento (sección 3).

| # | Actividad del enunciado | Sección |
|---|---|---|
| 1 | Pipeline de audio a espectrogramas normalizados y de tamaño consistente, listos para AST | 1 y 2 |
| 2 | *Fine-tuning* de AST durante 2 épocas | 5 a 9 |
| 3 | Precision, recall y F1; mejores y peores categorías; matriz de confusión y categorías más confundidas | 10 y 11 |

Todas las cifras del informe provienen de las salidas de este notebook.

## 0. Configuración

Todo el código está en este notebook para que sea autocontenido. Las decisiones de diseño principales son:
- **Espectrogramas calculados una sola vez** y guardados en `data/cache/` (no versionado), para no recalcularlos en cada época ni en cada experimento.
- **Resultados de cada corrida guardados** en `outputs/tables/runs/`. Si ya existen, se cargan en lugar de reentrenar.
- **Sin procesos auxiliares en el `DataLoader`** (`num_workers=0`), lo que evita errores de multiproceso de Jupyter en Windows y macOS. No afecta la velocidad, porque los espectrogramas ya están precalculados.
- **Audio leído con `soundfile` y remuestreado con `scipy`**, sin depender de `torchcodec` ni de FFmpeg (que `datasets` ≥ 4 exige para decodificar audio y que suele fallar en Windows).

**Si la GPU se queda sin memoria** (`CUDA out of memory`): use `batch_size=4, grad_accum=4` (mismo lote efectivo) o active `grad_checkpointing=True`.

In [ ]:
import os, io, re, json, math, time, random, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import ASTFeatureExtractor, ASTForAudioClassification, ASTConfig, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix, classification_report
from sklearn.manifold import TSNE
from scipy.fft import dct

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

ROOT = Path.cwd().resolve()
while not (ROOT / "README.md").exists() or not (ROOT / "notebooks").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Abra el notebook desde el repositorio esc50-ast-sound-classifier.")
    ROOT = ROOT.parent

QUICK = os.environ.get("MP4_QUICK", "0") == "1"   # prueba de humo sin internet: audio sintético y AST diminuto
RETRAIN = False                                   # True: reentrena aunque existan resultados guardados

OUT = ROOT / "outputs" / ("_smoke_test" if QUICK else "")
TAB, FIG, MODELS, RUNS = OUT / "tables", OUT / "figures", OUT / "models", OUT / "tables" / "runs"
SPLITS = ROOT / "data" / ("splits_smoke" if QUICK else "splits")
CACHE = ROOT / "data" / "cache" / ("smoke" if QUICK else "esc50")
REPORT_FIGS = (OUT / "report_figs") if QUICK else (ROOT / "report" / "figs")
for d in [TAB, FIG, MODELS, RUNS, SPLITS, CACHE, REPORT_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = dict(
    dataset="confit/esc50-demo", dataset_config="fold1",
    model="MIT/ast-finetuned-audioset-10-10-0.4593",
    sampling_rate=16_000,       # AST fue preentrenado con audio a 16 kHz
    epochs=2,                   # fijado por el enunciado
    batch_size=8, grad_accum=2, # lote efectivo de 16
    lr_backbone=3e-5,           # tasa baja para no destruir lo aprendido en AudioSet
    lr_head=1e-3,               # la capa nueva parte de cero y necesita una tasa mayor
    weight_decay=0.01, warmup=0.1, clip=1.0,
    specaug=dict(freq_masks=2, freq_width=24, time_masks=2, time_width=96),  # receta de AST para ESC-50 [2]
    val_folds=5,                # 1/5 de los grupos de entrenamiento va a validación
    grad_checkpointing=False,   # True si la GPU se queda sin memoria (más lento, usa mucha menos memoria)
    seed=42, seeds=[42, 7, 123],
)
if QUICK:
    CONFIG.update(epochs=1, batch_size=16, grad_accum=1, seeds=[42, 7])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"   # precisión mixta solo en CUDA; en MPS y CPU se usa fp32

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else ("Apple GPU (MPS)" if DEVICE.type == "mps" else "sin GPU")
print(f"Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"Dispositivo: {DEVICE} ({gpu}) | modo rápido: {QUICK}")
if DEVICE.type == "cpu" and not QUICK:
    print("ADVERTENCIA: sin GPU (CUDA o MPS) el fine-tuning de AST tarda varias horas.")
print(json.dumps(CONFIG, indent=1))

## 1. Carga y exploración de los datos

Se carga el fold 1 de ESC-50 desde Hugging Face con el código del enunciado. Los WAV originales (44,1 kHz) se leen sin decodificar (`Audio(decode=False)`) y la función `load_audio` los convierte a mono y los **remuestrea a 16 kHz**, la frecuencia con la que se preentrenó AST.

La función `describe_columns` identifica automáticamente las columnas de etiqueta, nombre de clase, fold y grabación de origen. Si la versión del dataset trae nombres distintos, basta con ajustar `COLS`.

Las 50 clases se agrupan en las 5 categorías oficiales de ESC-50 [1].

In [ ]:
CATEGORIAS = {
    "Animales": ["dog", "rooster", "pig", "cow", "frog", "cat", "hen", "insects", "sheep", "crow"],
    "Naturales y agua": ["rain", "sea_waves", "crackling_fire", "crickets", "chirping_birds", "water_drops", "wind",
                         "pouring_water", "toilet_flush", "thunderstorm"],
    "Humanos no verbales": ["crying_baby", "sneezing", "clapping", "breathing", "coughing", "footsteps", "laughing",
                            "brushing_teeth", "snoring", "drinking_sipping"],
    "Domésticos": ["door_wood_knock", "mouse_click", "keyboard_typing", "door_wood_creaks", "can_opening",
                   "washing_machine", "vacuum_cleaner", "clock_alarm", "clock_tick", "glass_breaking"],
    "Urbanos": ["helicopter", "chainsaw", "siren", "car_horn", "engine", "train", "church_bells", "airplane",
                "fireworks", "hand_saw"],
}
CLASE2CAT = {c: cat for cat, cs in CATEGORIAS.items() for c in cs}
norm_name = lambda s: re.sub(r"[\s\-]+", "_", str(s).strip().lower())

def synthetic_esc50(n_train_per_class=6, n_test_per_class=2, seconds=5.0, sr=44_100, seed=0):
    # Solo para la prueba de humo: tonos con frecuencia propia por clase + ruido, guardados como WAV de 44,1 kHz
    # (igual que los originales de ESC-50), para probar la misma ruta de lectura y remuestreo que los datos reales
    rng = np.random.default_rng(seed); classes = [c for cs in CATEGORIAS.values() for c in cs]; rows = {"train": [], "test": []}
    t = np.arange(int(seconds * sr)) / sr
    for k, c in enumerate(classes):
        for split, n, folds in [("train", n_train_per_class, [2, 3, 4, 5]), ("test", n_test_per_class, [1])]:
            for j in range(n):
                f0 = 150 + 60 * k
                wav = 0.3 * np.sin(2 * np.pi * f0 * t) + 0.05 * rng.standard_normal(t.size)
                src = f"{k:02d}{j // 2:03d}{int(split == 'test')}"          # 2 clips (tomas A y B) por grabación
                buf = io.BytesIO(); sf.write(buf, wav.astype(np.float32), sr, format="WAV", subtype="PCM_16")
                rows[split].append({"audio": {"bytes": buf.getvalue(), "path": f"{folds[j % len(folds)]}-{src}-{'AB'[j % 2]}-{k}.wav"},
                                    "target": k, "category": c, "fold": folds[j % len(folds)]})
    return rows["train"], rows["test"]

if QUICK:
    train_ds, test_ds = synthetic_esc50()
    columns = list(train_ds[0].keys())
else:
    from datasets import load_dataset, Audio
    train_ds = load_dataset(CONFIG["dataset"], CONFIG["dataset_config"], split="train")
    test_ds = load_dataset(CONFIG["dataset"], CONFIG["dataset_config"], split="test")
    print(train_ds)
    columns = train_ds.column_names
    audio_col = next(c for c in columns if c.lower() in ("audio", "wav", "sound"))
    train_ds = train_ds.cast_column(audio_col, Audio(decode=False))   # bytes del WAV original; se decodifican con load_audio
    test_ds = test_ds.cast_column(audio_col, Audio(decode=False))

def load_audio(a, target_sr=CONFIG["sampling_rate"]):
    # Devuelve (forma de onda mono float32, frecuencia) a target_sr, para audio ya decodificado o bytes / ruta de un WAV
    if isinstance(a, dict) and a.get("array") is not None:
        x, sr = np.asarray(a["array"], dtype=np.float32), int(a["sampling_rate"])
        x = x.mean(axis=0) if x.ndim > 1 else x
    elif isinstance(a, dict):
        x, sr = sf.read(io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"], dtype="float32", always_2d=True)
        x = x.mean(axis=1)
    else:                                   # AudioDecoder de torchcodec (datasets >= 4 con decode=True)
        smp = a.get_all_samples(); x, sr = smp.data.numpy().mean(axis=0), int(smp.sample_rate)
    if target_sr and sr != target_sr:
        g = math.gcd(sr, target_sr); x = resample_poly(x, target_sr // g, sr // g)
    return np.ascontiguousarray(x, dtype=np.float32), (target_sr or sr)

def describe_columns(columns, sample):
    pick = lambda opts: next((c for c in columns if c.lower() in opts), None)
    cols = dict(audio=pick(("audio", "wav", "sound")), label=pick(("target", "label", "labels", "class_id")),
                name=pick(("category", "class", "class_name", "label_name")), fold=pick(("fold",)),
                src=pick(("src_file", "source", "source_file")), filename=pick(("filename", "file", "path")))
    return cols
COLS = describe_columns(columns, train_ds[0])
print("Columnas:", columns, "\nAsignación:", COLS)
assert COLS["audio"] and (COLS["label"] or COLS["name"]), "Ajuste COLS: no se encontró la columna de audio o de etiqueta"

In [ ]:
def audio_info(a):
    # Duración y frecuencia original sin decodificar el audio cuando se tienen los bytes
    if isinstance(a, dict) and a.get("array") is None:
        inf = sf.info(io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"])
        return dict(duracion_s=inf.duration, sr_original=inf.samplerate, canales=inf.channels)
    x, sr = load_audio(a, target_sr=None) if not isinstance(a, dict) else (np.asarray(a["array"]), a["sampling_rate"])
    return dict(duracion_s=x.shape[-1] / sr, sr_original=sr, canales=1 if x.ndim == 1 else x.shape[0])

def build_metadata(ds, split):
    # Una fila por clip con etiqueta, nombre de clase, categoría, fold y grabación de origen (sin el audio)
    names = None
    if not QUICK and COLS["label"] and hasattr(ds.features[COLS["label"]], "names"):
        names = ds.features[COLS["label"]].names
    rows = []
    for i in range(len(ds)):
        ex = ds[i]
        a = ex[COLS["audio"]]
        fname = ex.get(COLS["filename"]) if COLS["filename"] else None
        fname = fname or (a.get("path") if isinstance(a, dict) else None) or ""
        fname = os.path.basename(str(fname))
        m = re.match(r"(\d+)-(\w+)-([A-Z])-(\d+)\.wav$", fname)       # formato oficial: {fold}-{src}-{take}-{target}.wav
        name = ex[COLS["name"]] if COLS["name"] else (names[ex[COLS["label"]]] if names else None)
        label = ex[COLS["label"]] if COLS["label"] else None
        rows.append(dict(idx=i, split_hf=split, archivo=fname, clase=norm_name(name) if name is not None else str(label),
                         label_orig=label, fold=ex[COLS["fold"]] if COLS["fold"] else (int(m.group(1)) if m else None),
                         src=ex[COLS["src"]] if COLS["src"] else (m.group(2) if m else f"{split}_{i}"),
                         **audio_info(a)))
    return pd.DataFrame(rows)

t0 = time.time()
meta = pd.concat([build_metadata(train_ds, "train"), build_metadata(test_ds, "test")], ignore_index=True)
CLASSES = sorted(meta.clase.unique(), key=lambda c: (list(CATEGORIAS).index(CLASE2CAT.get(c, "Urbanos")), c))
C2I = {c: i for i, c in enumerate(CLASSES)}
meta["label"] = meta.clase.map(C2I); meta["categoria"] = meta.clase.map(CLASE2CAT)
sin_cat = sorted(set(meta.clase[meta.categoria.isna()]))
print(f"Clips: {len(meta)} | clases: {len(CLASSES)} | metadatos en {time.time() - t0:.0f} s")
print("Clases sin categoría conocida (revisar nombres):", sin_cat or "ninguna")
display(meta.groupby("split_hf").agg(clips=("idx", "size"), clases=("clase", "nunique"), folds=("fold", lambda s: sorted(s.dropna().unique())),
                                     grabaciones=("src", "nunique"), duracion_s_min=("duracion_s", "min"),
                                     duracion_s_max=("duracion_s", "max"), sr_original=("sr_original", "first"), canales=("canales", "max")))
display(pd.crosstab(meta.categoria, meta.split_hf).assign(clases=meta.groupby("categoria").clase.nunique()))

In [ ]:
# Ejemplo por categoría: forma de onda y espectrograma log-mel (el mismo que recibe AST, sección 2)
fe_vis = ASTFeatureExtractor() if QUICK else ASTFeatureExtractor.from_pretrained(CONFIG["model"])
ejemplos = meta[meta.split_hf == "train"].groupby("categoria").head(1)
fig, axes = plt.subplots(2, len(ejemplos), figsize=(3.2 * len(ejemplos), 5))
for k, (_, r) in enumerate(ejemplos.iterrows()):
    wav, sr = load_audio(train_ds[int(r.idx)][COLS["audio"]])
    axes[0, k].plot(np.arange(wav.size) / sr, wav, lw=0.4); axes[0, k].set_title(f"{r.categoria}\n{r.clase}", fontsize=9)
    axes[0, k].set_xlabel("s")
    spec = fe_vis(wav, sampling_rate=sr, return_tensors="np")["input_values"][0]
    axes[1, k].imshow(spec.T, origin="lower", aspect="auto", cmap="magma"); axes[1, k].set_xlabel("trama (10 ms)")
axes[1, 0].set_ylabel("banda mel")
plt.tight_layout(); plt.savefig(FIG / "eda_ejemplos_por_categoria.png", dpi=150); plt.show()

## 2. Pipeline de audio a espectrogramas

Cada clip pasa por el `ASTFeatureExtractor` del modelo preentrenado, que reproduce exactamente el preprocesamiento usado en su entrenamiento [2]:

1. **Audio mono a 16 kHz** (remuestreado al cargar).
2. **Banco de filtros log-mel** de 128 bandas al estilo Kaldi: ventanas de 25 ms cada 10 ms, es decir, 100 tramas por segundo.
3. **Tamaño fijo de 1 024 tramas** (10,24 s): los clips de ESC-50 duran 5 s (≈ 500 tramas), así que se rellenan con ceros hasta 1 024. Así todas las entradas tienen la forma 1 024 × 128 que espera AST, y se conservan los *embeddings* de posición aprendidos en AudioSet (cambiar la longitud obligaría a interpolarlos).
4. **Normalización** con la media (−4,27) y el doble de la desviación estándar (4,57) de AudioSet, las mismas del preentrenamiento. Usar otra normalización cambiaría la distribución de entrada con la que se aprendieron los pesos.

El resultado es una "imagen" de 1 024 × 128 por clip, que AST divide en parches de 16 × 16 (sección 5). Los espectrogramas se calculan una vez y se guardan en `data/cache/` como `float16`.

In [ ]:
FE = ASTFeatureExtractor() if QUICK else ASTFeatureExtractor.from_pretrained(CONFIG["model"])
print(FE)

def compute_features(ds, split):
    path = CACHE / f"{split}_input_values.npy"
    n = len(ds)
    if path.exists() and not RETRAIN:
        arr = np.load(path, mmap_mode="r")
        if arr.shape[0] == n:
            return arr
    arr = np.lib.format.open_memmap(path, mode="w+", dtype=np.float16, shape=(n, FE.max_length, FE.num_mel_bins))
    t0 = time.time()
    for i in range(n):
        wav, sr = load_audio(ds[i][COLS["audio"]])
        arr[i] = FE(wav, sampling_rate=sr, return_tensors="np")["input_values"][0]
        if (i + 1) % 400 == 0:
            print(f"  {split}: {i + 1}/{n} ({time.time() - t0:.0f} s)")
    arr.flush()
    return np.load(path, mmap_mode="r")

X = {s: compute_features(ds, s) for s, ds in [("train", train_ds), ("test", test_ds)]}
for s, a in X.items():
    muestra = np.asarray(a[:64], dtype=np.float32)
    print(f"{s}: {a.shape} {a.dtype} | media {muestra.mean():.3f} | desv. {muestra.std():.3f} | rango [{muestra.min():.2f}, {muestra.max():.2f}]")
assert all(a.shape[1:] == (FE.max_length, FE.num_mel_bins) for a in X.values()), "Tamaños inconsistentes"
pad_val = (0 - FE.mean) / (2 * FE.std)
frac_pad = float(np.mean(np.isclose(X["train"][:64].astype(np.float32), pad_val, atol=1e-3).all(axis=2)))
print(f"Fracción de tramas de relleno (estimada en 64 clips): {frac_pad:.1%}")

## 3. Partición entrenamiento / validación / prueba

- **Prueba:** el split `test` del fold 1 (400 clips), intacto hasta el reporte final.
- **Entrenamiento y validación:** el split `train` (folds 2 a 5) se divide con `StratifiedGroupKFold`: ~80 % entrenamiento y ~20 % validación, estratificado por clase y **agrupado por grabación de origen**. En ESC-50, varios clips pueden venir de la misma grabación (`src_file`), y si unos quedaran en entrenamiento y otros en validación, la validación sobrestimaría el desempeño. Los folds oficiales ya respetan esta regla entre entrenamiento y prueba [1]; aquí se verifica y se aplica también a la validación.

La partición y los metadatos se guardan en `data/splits/` para que la ejecución sea reproducible.

In [ ]:
SPLIT_PATH = SPLITS / "split.csv"
tr_meta = meta[meta.split_hf == "train"].copy()
if SPLIT_PATH.exists() and not RETRAIN:
    sp = pd.read_csv(SPLIT_PATH)
    meta = meta.drop(columns=[c for c in ["split"] if c in meta]).merge(sp[["split_hf", "idx", "split"]], on=["split_hf", "idx"])
    print("Partición cargada desde", SPLIT_PATH.relative_to(ROOT))
else:
    sg = StratifiedGroupKFold(n_splits=CONFIG["val_folds"], shuffle=True, random_state=CONFIG["seed"])
    tr_i, va_i = next(sg.split(tr_meta, tr_meta.label, tr_meta.src.astype(str)))
    meta["split"] = "test"
    meta.loc[tr_meta.index[tr_i], "split"] = "train"; meta.loc[tr_meta.index[va_i], "split"] = "val"
    meta.to_csv(SPLIT_PATH, index=False)
    print("Partición creada y guardada en", SPLIT_PATH.relative_to(ROOT))

src_sets = {s: set(meta.loc[meta.split == s, "src"].astype(str)) for s in ["train", "val", "test"]}
cruces = {f"{a}-{b}": len(src_sets[a] & src_sets[b]) for a, b in [("train", "val"), ("train", "test"), ("val", "test")]}
print("Grabaciones de origen compartidas entre particiones:", cruces)
assert cruces["train-val"] == 0, "La validación comparte grabaciones con el entrenamiento"
tabla = meta.groupby("split").agg(clips=("idx", "size"), clases=("clase", "nunique"),
                                  min_por_clase=("clase", lambda s: s.value_counts().min()),
                                  max_por_clase=("clase", lambda s: s.value_counts().max())).loc[["train", "val", "test"]]
display(tabla); tabla.to_csv(TAB / "particion.csv")

IDX = {s: meta.loc[meta.split == s].sort_values("idx") for s in ["train", "val", "test"]}
SRC_ARR = {"train": "train", "val": "train", "test": "test"}   # de qué arreglo de espectrogramas sale cada partición
Y = {s: IDX[s].label.values for s in IDX}

## 4. Línea base: MFCC + regresión logística

Para saber cuánto aporta AST se entrena un modelo clásico: los **MFCC** [10] (transformada discreta del coseno del espectrograma log-mel, que lo resume en pocos coeficientes decorrelacionados) de cada clip, resumidos con su media y desviación estándar en el tiempo (solo tramas con audio, sin relleno), y una regresión logística multinomial. La regularización C se elige en validación.

In [ ]:
def mfcc_stats(arr, n_mfcc=20):
    feats = np.zeros((arr.shape[0], 4 * n_mfcc), dtype=np.float32)
    for i in range(arr.shape[0]):
        s = arr[i].astype(np.float32)
        s = s[~np.isclose(s, pad_val, atol=1e-3).all(axis=1)] if (~np.isclose(s, pad_val, atol=1e-3).all(axis=1)).sum() > 2 else s
        c = dct(s, type=2, axis=1, norm="ortho")[:, :n_mfcc]
        d = np.diff(c, axis=0)
        feats[i] = np.concatenate([c.mean(0), c.std(0), d.mean(0), d.std(0)])
    return feats

BASE_PATH = TAB / "linea_base_mfcc.json"
if BASE_PATH.exists() and not RETRAIN:
    BASE = json.loads(BASE_PATH.read_text()); P_BASE = np.load(TAB / "linea_base_mfcc_prob_test.npy")
    print("Línea base cargada")
else:
    Fm = {s: mfcc_stats(X[SRC_ARR[s]][IDX[s].idx.values]) for s in IDX}
    sc = StandardScaler().fit(Fm["train"]); Fz = {s: sc.transform(v) for s, v in Fm.items()}
    grid = [(f1_score(Y["val"], LogisticRegression(C=C, max_iter=5000).fit(Fz["train"], Y["train"]).predict(Fz["val"]), average="macro"), C)
            for C in [0.01, 0.1, 1.0, 10.0]]
    vf, C = max(grid)
    clf = LogisticRegression(C=C, max_iter=5000).fit(Fz["train"], Y["train"])
    P_BASE = clf.predict_proba(Fz["test"])
    pr, rc, f1, _ = precision_recall_fscore_support(Y["test"], P_BASE.argmax(1), average="macro", zero_division=0)
    BASE = dict(modelo="MFCC (media y desv.) + reg. logística", C=C, val_f1_macro=vf, test_accuracy=accuracy_score(Y["test"], P_BASE.argmax(1)),
                test_precision_macro=pr, test_recall_macro=rc, test_f1_macro=f1)
    BASE_PATH.write_text(json.dumps(BASE, indent=1)); np.save(TAB / "linea_base_mfcc_prob_test.npy", P_BASE)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in BASE.items()})

## 5. Arquitectura

```
Espectrograma log-mel 1 024 × 128
 └─ Parches 16 × 16 con solape (paso 10): 101 × 12 = 1 212 parches → proyección lineal a 768
 └─ + token [CLS] + token de destilación + embeddings de posición
 └─ 12 bloques transformer (auto-atención multi-cabeza, 12 cabezas, 768 dimensiones)
 └─ Promedio de los tokens [CLS] y de destilación → LayerNorm → Densa 768 → 50 → softmax
```

**Justificación**
- **AST [2]:** trata el espectrograma como una imagen y aplica un *Vision Transformer* [4] sin convoluciones. La **auto-atención** [3] relaciona cualquier par de parches del espectrograma desde la primera capa, por lo que capta patrones de largo alcance en tiempo y frecuencia (por ejemplo, un canto de ave que se repite varias veces en el clip).
- **Pesos preentrenados en AudioSet** [5] (2 millones de clips, 527 clases), y antes en ImageNet a través de DeiT [6]. AudioSet incluye sonidos muy parecidos a los de ESC-50, así que las representaciones ya distinguen gran parte de las clases y 2 épocas bastan para adaptarlas.
- **Parches con solape** (paso 10 en lugar de 16): el modelo original se preentrenó así, y conservar esa configuración permite reutilizar sus pesos tal cual.
- **Cabeza nueva LayerNorm + Densa 768 → 50:** reemplaza a la de 527 clases de AudioSet (`ignore_mismatched_sizes=True` la reinicia). Es la única parte que parte de cero (38 450 parámetros).
- **Softmax con entropía cruzada:** cada clip tiene una sola clase, así que la salida debe ser una distribución de probabilidad sobre las 50 clases (sección 6).

In [ ]:
def build_model(freeze_backbone=False):
    if QUICK:
        cfg = ASTConfig(hidden_size=64, num_hidden_layers=2, num_attention_heads=2, intermediate_size=128,
                        num_labels=len(CLASSES), max_length=FE.max_length, num_mel_bins=FE.num_mel_bins)
        m = ASTForAudioClassification(cfg)
    else:
        m = ASTForAudioClassification.from_pretrained(CONFIG["model"], num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)),
                                                      label2id=C2I, ignore_mismatched_sizes=True)
    if CONFIG["grad_checkpointing"] and not freeze_backbone:
        m.gradient_checkpointing_enable()
    if freeze_backbone:
        for n, p in m.named_parameters():
            p.requires_grad = n.startswith("classifier")
    return m

m = build_model()
n_tot = sum(p.numel() for p in m.parameters()); n_head = sum(p.numel() for n, p in m.named_parameters() if n.startswith("classifier"))
n_patches = m.audio_spectrogram_transformer.embeddings.patch_embeddings.projection
print(f"{type(m).__name__} | parámetros: {n_tot:,} | cabeza nueva: {n_head:,}")
print("Proyección de parches:", n_patches, "| tokens por clip:", m.audio_spectrogram_transformer.embeddings.position_embeddings.shape[1])
print(m.classifier)
del m

## 6. Entrenamiento y evaluación

- **Pérdida: entropía cruzada categórica.** Minimizarla equivale a maximizar la verosimilitud de la distribución categórica que produce la softmax [7]. Junto con la softmax, su gradiente respecto a los logits es *p − y*: no se satura como el del error cuadrático y castiga con fuerza los errores cometidos con alta confianza.
- **Optimizador:** AdamW [8] con dos tasas de aprendizaje: 3·10⁻⁵ para el transformer preentrenado (cambios pequeños, para no olvidar lo aprendido en AudioSet) y 10⁻³ para la cabeza nueva, que parte de cero. *Weight decay* 0,01, *warmup* lineal del 10 % y decaimiento lineal.
- **2 épocas** (fijadas por el enunciado), lote efectivo de 16 (8 × 2 pasos de acumulación), *gradient clipping* con norma 1 y precisión mixta en GPU NVIDIA. Se reporta el modelo al final de la segunda época, y la validación se mide al cierre de cada época.
- **SpecAugment** [9] (experimento 2): durante el entrenamiento se borran al azar 2 franjas de hasta 24 bandas de frecuencia y 2 franjas de hasta 96 tramas, la configuración de AST para ESC-50 [2]. Es un aumento de datos que obliga al modelo a no depender de una sola región del espectrograma.
- **Métricas:** *accuracy* y precision, recall y F1 **macro** (promedio sin ponderar de las 50 clases), y las mismas métricas por clase y por categoría.

In [ ]:
class SpecDataset(Dataset):
    def __init__(self, split):
        self.arr, self.idx, self.y = X[SRC_ARR[split]], IDX[split].idx.values, Y[split]
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        return torch.from_numpy(np.asarray(self.arr[self.idx[i]], dtype=np.float32)), int(self.y[i])

def make_loader(split, shuffle, seed):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(SpecDataset(split), batch_size=CONFIG["batch_size"], shuffle=shuffle, num_workers=0,
                      generator=g if shuffle else None, pin_memory=DEVICE.type == "cuda")

def spec_augment(x, cfg):
    # x: (B, T, F). Borra franjas aleatorias de frecuencia y de tiempo (valor 0 = media tras la normalización)
    B, T, Fq = x.shape; x = x.clone()
    for _ in range(cfg["freq_masks"]):
        w = torch.randint(0, cfg["freq_width"] + 1, (B,), device=x.device); f0 = (torch.rand(B, device=x.device) * (Fq - w)).long()
        ar = torch.arange(Fq, device=x.device)[None, :]
        x = x.masked_fill(((ar >= f0[:, None]) & (ar < (f0 + w)[:, None]))[:, None, :], 0.0)
    for _ in range(cfg["time_masks"]):
        w = torch.randint(0, cfg["time_width"] + 1, (B,), device=x.device); t0 = (torch.rand(B, device=x.device) * (T - w)).long()
        ar = torch.arange(T, device=x.device)[None, :]
        x = x.masked_fill(((ar >= t0[:, None]) & (ar < (t0 + w)[:, None]))[:, :, None], 0.0)
    return x

def compute_metrics(y, prob):
    pred = np.asarray(prob).argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return dict(accuracy=accuracy_score(y, pred), precision_macro=p, recall_macro=r, f1_macro=f)

@torch.no_grad()
def predict(model, loader, return_emb=False):
    model.eval(); probs, embs = [], []
    for xb, _ in loader:
        xb = xb.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(input_values=xb, output_hidden_states=return_emb)
        probs.append(out.logits.float().softmax(-1).cpu())
        if return_emb:
            h = model.audio_spectrogram_transformer.layernorm(out.hidden_states[-1])
            embs.append(((h[:, 0] + h[:, 1]) / 2).float().cpu())   # representación que recibe la cabeza
    return (torch.cat(probs).numpy(), torch.cat(embs).numpy()) if return_emb else torch.cat(probs).numpy()

In [ ]:
RESULTS, HIST, PV, PT = [], {}, {}, {}

def run_experiment(strategy, augment, seed):
    tag = f"{strategy}{'_specaug' if augment else ''}_s{seed}"
    paths = {k: RUNS / f"{tag}{s}" for k, s in [("res", ".json"), ("hist", "_historia.csv"), ("pv", "_prob_val.npy"),
                                                ("pt", "_prob_test.npy"), ("emb", "_emb_test.npy")]}
    if paths["res"].exists() and not RETRAIN:
        res = json.loads(paths["res"].read_text()); hist = pd.read_csv(paths["hist"])
        print(f"[{tag}] resultados cargados (entrenado en {res['tiempo_min']:.1f} min en {res['dispositivo']}). Historia:")
        display(hist.round(4))
    else:
        set_seed(seed)
        tr_dl, va_dl, te_dl = make_loader("train", True, seed), make_loader("val", False, seed), make_loader("test", False, seed)
        model = build_model(freeze_backbone=(strategy == "solo_cabeza")).to(DEVICE)
        head = [p for n, p in model.named_parameters() if n.startswith("classifier") and p.requires_grad]
        body = [p for n, p in model.named_parameters() if not n.startswith("classifier") and p.requires_grad]
        groups = [{"params": head, "lr": CONFIG["lr_head"]}] + ([{"params": body, "lr": CONFIG["lr_backbone"]}] if body else [])
        opt = torch.optim.AdamW(groups, weight_decay=CONFIG["weight_decay"])
        steps = int(np.ceil(len(tr_dl) / CONFIG["grad_accum"])) * CONFIG["epochs"]
        sched = get_linear_schedule_with_warmup(opt, int(CONFIG["warmup"] * steps), steps)
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        hist, t_start = [], time.time()
        for ep in range(1, CONFIG["epochs"] + 1):
            model.train(); t0, tl, tc, tn = time.time(), 0.0, 0, 0
            for i, (xb, yb) in enumerate(tr_dl):
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                if augment:
                    xb = spec_augment(xb, CONFIG["specaug"])
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                    logits = model(input_values=xb).logits
                    loss = F.cross_entropy(logits.float(), yb)
                scaler.scale(loss / CONFIG["grad_accum"]).backward()
                if (i + 1) % CONFIG["grad_accum"] == 0 or i + 1 == len(tr_dl):
                    scaler.unscale_(opt); nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], CONFIG["clip"])
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                tl += loss.item() * len(yb); tc += (logits.argmax(-1) == yb).sum().item(); tn += len(yb)
            pv = predict(model, va_dl); mv = compute_metrics(Y["val"], pv)
            vloss = float(F.nll_loss(torch.log(torch.tensor(pv).clamp_min(1e-9)), torch.tensor(Y["val"])))
            hist.append(dict(epoch=ep, train_loss=tl / tn, train_acc=tc / tn, val_loss=vloss, val_acc=mv["accuracy"],
                             val_f1_macro=mv["f1_macro"], seg=time.time() - t0))
            print(f"[{tag}] ép {ep} | train loss {tl/tn:.4f} acc {tc/tn:.4f} | val loss {vloss:.4f} acc {mv['accuracy']:.4f} "
                  f"F1-macro {mv['f1_macro']:.4f} | {time.time() - t0:.0f}s")
        pt, emb = predict(model, te_dl, return_emb=True)
        hist = pd.DataFrame(hist); last = hist.iloc[-1]
        res = dict(tag=tag, estrategia=strategy, specaug=augment, seed=seed, epocas=len(hist),
                   train_acc=float(last.train_acc), val_acc=float(last.val_acc), val_f1_macro=float(last.val_f1_macro),
                   parametros_entrenables=int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
                   tiempo_min=(time.time() - t_start) / 60, dispositivo=gpu,
                   **{f"test_{k}": float(v) for k, v in compute_metrics(Y["test"], pt).items()})
        hist.to_csv(paths["hist"], index=False); np.save(paths["pv"], pv); np.save(paths["pt"], pt)
        np.save(paths["emb"], emb.astype(np.float16)); paths["res"].write_text(json.dumps(res, indent=1))
        del model, opt
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        elif DEVICE.type == "mps":
            torch.mps.empty_cache()
    HIST[tag], PV[tag], PT[tag] = hist, np.load(paths["pv"]), np.load(paths["pt"])
    RESULTS[:] = [r for r in RESULTS if r["tag"] != tag] + [res]
    return res

SHOW = ["estrategia", "specaug", "seed", "parametros_entrenables", "train_acc", "val_f1_macro", "test_accuracy",
        "test_precision_macro", "test_recall_macro", "test_f1_macro", "tiempo_min"]

## 7. Experimento 1: *fine-tuning* completo frente a solo la cabeza

Con la misma partición, semilla (42) y 2 épocas, se comparan dos estrategias:
- **Solo la cabeza:** el transformer queda congelado y solo se entrena la capa nueva. Mide qué tan buenas son, sin adaptar, las representaciones aprendidas en AudioSet.
- **Fine-tuning completo:** se entrenan todas las capas, con la tasa baja en el transformer.

La estrategia final se elige por **F1-macro de validación**.

In [ ]:
for strategy in ["solo_cabeza", "completo"]:
    run_experiment(strategy, augment=False, seed=CONFIG["seed"])
e1 = pd.DataFrame([r for r in RESULTS if not r["specaug"] and r["seed"] == CONFIG["seed"]])
display(e1[SHOW].round(4)); e1[SHOW].to_csv(TAB / "exp1_estrategias.csv", index=False)
BEST_STRATEGY = e1.loc[e1.val_f1_macro.idxmax(), "estrategia"]
print("Estrategia elegida por F1-macro de validación:", BEST_STRATEGY)

## 8. Experimento 2: aumento de datos con SpecAugment

Con la estrategia elegida se compara entrenar con y sin SpecAugment. Con solo 1 280 clips de entrenamiento y un modelo de 86 M parámetros, el aumento de datos puede reducir el sobreajuste. Con solo 2 épocas, también puede quedarse corto en tiempo de entrenamiento. Se decide por F1-macro de validación.

In [ ]:
run_experiment(BEST_STRATEGY, augment=True, seed=CONFIG["seed"])
e2 = pd.DataFrame([r for r in RESULTS if r["estrategia"] == BEST_STRATEGY and r["seed"] == CONFIG["seed"]])
display(e2[SHOW].round(4)); e2[SHOW].to_csv(TAB / "exp2_specaugment.csv", index=False)
BEST_AUG = bool(e2.loc[e2.val_f1_macro.idxmax(), "specaug"])
FINAL = dict(strategy=BEST_STRATEGY, augment=BEST_AUG)
print("Configuración final:", FINAL)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for r in RESULTS:
    if r["seed"] != CONFIG["seed"]:
        continue
    h = HIST[r["tag"]]; lab = f"{r['estrategia']}{' + SpecAugment' if r['specaug'] else ''}"
    c = ax[0].plot(h.epoch, h.val_loss, marker="o", label=lab)[0].get_color()
    ax[0].plot(h.epoch, h.train_loss, ls="--", color=c, alpha=0.6); ax[1].plot(h.epoch, h.val_f1_macro, marker="o", color=c, label=lab)
ax[0].set_title("Pérdida (punteada: train)"); ax[1].set_title("F1-macro en validación")
for a in ax:
    a.set_xlabel("Época"); a.set_xticks(range(1, CONFIG["epochs"] + 1)); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "exp1_exp2_curvas.png", dpi=150); plt.show()

## 9. Experimento 3: variabilidad entre semillas

La configuración final se reentrena con otras dos semillas (cambian la inicialización de la cabeza, el orden de los lotes y las máscaras de SpecAugment). Se reporta **media ± desviación estándar** en prueba.

In [ ]:
for seed in CONFIG["seeds"]:
    run_experiment(FINAL["strategy"], FINAL["augment"], seed)
R = pd.DataFrame(RESULTS)
e3 = R[(R.estrategia == FINAL["strategy"]) & (R.specaug == FINAL["augment"])].sort_values("seed")
mcols = ["test_accuracy", "test_precision_macro", "test_recall_macro", "test_f1_macro"]
display(e3[["seed", "val_f1_macro"] + mcols].round(4))
agg = e3[mcols].agg(["mean", "std"]).T
agg["media ± std"] = agg["mean"].map("{:.4f}".format) + " ± " + agg["std"].map("{:.4f}".format)
display(agg[["media ± std"]]); agg.to_csv(TAB / "exp3_semillas.csv")
FINAL_TAG = f"{FINAL['strategy']}{'_specaug' if FINAL['augment'] else ''}_s{CONFIG['seed']}"

## 10. Resultados cuantitativos: global, por clase y por categoría

Tabla resumen; precision, recall y F1 de **cada una de las 50 clases** del modelo final (semilla 42), con las 5 mejores y las 5 peores; métricas agregadas por **categoría**; e intervalos de confianza al 95 % por *bootstrap* (1 000 remuestreos del conjunto de prueba).

In [ ]:
rows = [dict(modelo=BASE["modelo"], **{k: BASE[f"test_{k}"] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]})]
for r in RESULTS:
    rows.append(dict(modelo=f"AST {r['estrategia']}{' + SpecAugment' if r['specaug'] else ''}, semilla {r['seed']}",
                     **{k: r[f"test_{k}"] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]}))
resumen = pd.DataFrame(rows).set_index("modelo")
resumen.loc[f"AST final ({FINAL_TAG.rsplit('_s', 1)[0]}): media de {len(e3)} semillas"] = e3[mcols].mean().values
resumen.loc["   desviación estándar"] = e3[mcols].std().values
display(resumen.round(4)); resumen.to_csv(TAB / "resumen_test.csv")

y = Y["test"]; P = PT[FINAL_TAG]; pred = P.argmax(1)
p, r_, f, s = precision_recall_fscore_support(y, pred, labels=range(len(CLASSES)), zero_division=0)
pc = pd.DataFrame({"categoria": [CLASE2CAT.get(c) for c in CLASSES], "precision": p, "recall": r_, "f1": f, "soporte": s}, index=CLASSES)
pc["errores"] = [int(((y == i) & (pred != i)).sum()) for i in range(len(CLASSES))]
pc.to_csv(TAB / "metricas_por_clase.csv")
print("Las 5 clases con mejor y peor F1 (prueba):")
display(pd.concat([pc.sort_values(["f1", "recall"], ascending=False).head(5), pc.sort_values(["f1", "recall"]).head(5)]).round(3))
print("Métricas por clase de las 50 clases:"); display(pc.round(3))

cat_names = list(CATEGORIAS)
yc = np.array([cat_names.index(CLASE2CAT[CLASSES[i]]) for i in y]); pcat = np.array([cat_names.index(CLASE2CAT[CLASSES[i]]) for i in pred])
pr_c, rc_c, f_c, s_c = precision_recall_fscore_support(yc, pcat, labels=range(5), zero_division=0)
por_cat = pd.DataFrame({"precision (categoría)": pr_c, "recall (categoría)": rc_c, "f1 (categoría)": f_c, "soporte": s_c,
                        "F1 medio de sus clases": pc.groupby("categoria").f1.mean().reindex(cat_names).values,
                        "errores de clase": pc.groupby("categoria").errores.sum().reindex(cat_names).values}, index=cat_names)
print("Desempeño por categoría (la predicción de clase se agrupa en su categoría):"); display(por_cat.round(3))
por_cat.to_csv(TAB / "metricas_por_categoria.csv")

rng = np.random.default_rng(0); pb = P_BASE.argmax(1)
boot = np.array([[f1_score(y[i], pred[i], average="macro"), f1_score(y[i], pb[i], average="macro")]
                 for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))])
F1_CI = np.percentile(boot[:, 0], [2.5, 97.5]); DIFF_CI = np.percentile(boot[:, 0] - boot[:, 1], [2.5, 97.5])
print(f"F1-macro final: {f1_score(y, pred, average='macro'):.4f} (IC 95 %: {F1_CI[0]:.4f} a {F1_CI[1]:.4f})")
print(f"Diferencia con la línea base MFCC: {f1_score(y, pred, average='macro') - f1_score(y, pb, average='macro'):+.4f} "
      f"(IC 95 %: {DIFF_CI[0]:+.4f} a {DIFF_CI[1]:+.4f})")

In [ ]:
# Matriz de confusión 50 × 50 (clases ordenadas por categoría) y 5 × 5 por categoría
cm = confusion_matrix(y, pred, labels=range(len(CLASSES)))
cmc = confusion_matrix(yc, pcat, labels=range(5))
fig, ax = plt.subplots(1, 2, figsize=(16, 7.2), gridspec_kw={"width_ratios": [2.2, 1]})
ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(len(CLASSES)), CLASSES, rotation=90, fontsize=6); ax[0].set_yticks(range(len(CLASSES)), CLASSES, fontsize=6)
for b in range(10, len(CLASSES), 10):
    ax[0].axhline(b - 0.5, color="k", lw=0.6); ax[0].axvline(b - 0.5, color="k", lw=0.6)
for i, j in zip(*np.nonzero(cm)):
    if i != j:
        ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=5, color="red")
ax[0].set_xlabel("Predicción"); ax[0].set_ylabel("Real"); ax[0].set_title("Matriz de confusión por clase (rojo: errores; líneas: categorías)")
ax[1].imshow(cmc / cmc.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        ax[1].text(j, i, cmc[i, j], ha="center", va="center", color="white" if cmc[i, j] / cmc[i].sum() > 0.5 else "black")
ax[1].set_xticks(range(5), cat_names, rotation=30, ha="right"); ax[1].set_yticks(range(5), cat_names)
ax[1].set_title("Matriz de confusión por categoría")
plt.tight_layout(); plt.savefig(FIG / "matrices_confusion.png", dpi=150); plt.show()

pares = pd.DataFrame([(CLASSES[i], CLASSES[j], int(cm[i, j]), CLASE2CAT[CLASSES[i]] == CLASE2CAT[CLASSES[j]])
                      for i, j in zip(*np.nonzero(cm)) if i != j], columns=["real", "predicha", "clips", "misma_categoria"])
pares = pares.sort_values("clips", ascending=False).reset_index(drop=True)
print(f"Errores: {int(cm.sum() - np.trace(cm))} de {int(cm.sum())} | dentro de la misma categoría: {int(pares[pares.misma_categoria].clips.sum())}")
print("Pares de clases más confundidos:"); display(pares.head(10)); pares.to_csv(TAB / "pares_confundidos.csv", index=False)

## 11. Resultados cualitativos

1. **Errores del modelo final** con su espectrograma, clase real, predicha y probabilidad. Ayudan a ver si las confusiones son acústicamente razonables (por ejemplo, sonidos de banda ancha parecidos como lluvia, viento y fritura).
2. **Espacio de representaciones:** t-SNE [11] de la representación que recibe la cabeza, con el transformer congelado (estrategia "solo cabeza") y tras el *fine-tuning* completo.

In [ ]:
TE = IDX["test"].reset_index(drop=True)
err = pd.DataFrame({"archivo": TE.archivo, "real": [CLASSES[i] for i in y], "predicha": [CLASSES[i] for i in pred],
                    "P(predicha)": P.max(1).round(3), "P(real)": P[np.arange(len(y)), y].round(3),
                    "MFCC": [CLASSES[i] for i in pb]})
err = err[err.real != err.predicha].sort_values("P(predicha)", ascending=False)
print(f"Errores del modelo final: {len(err)} | también fallados por MFCC: {int((err.MFCC != err.real).sum())}")
display(err.head(12)); err.to_csv(TAB / "errores_modelo_final.csv", index=False)

show = err.head(6)
fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3.2)) if len(show) else (None, [])
for a, (k, r) in zip(np.atleast_1d(axes), show.iterrows()):
    spec = np.asarray(X["test"][TE.idx[k]], dtype=np.float32)
    a.imshow(spec[:520].T, origin="lower", aspect="auto", cmap="magma")
    a.set_title(f"real: {r.real}\npred: {r.predicha} ({r['P(predicha)']:.2f})", fontsize=8); a.set_xticks([]); a.set_yticks([])
if len(show):
    plt.suptitle("Errores de mayor confianza (primeros 5,2 s del espectrograma)"); plt.tight_layout()
    plt.savefig(FIG / "errores_espectrogramas.png", dpi=150); plt.show()

In [ ]:
emb_tags = {"Transformer congelado": f"solo_cabeza_s{CONFIG['seed']}", "Tras fine-tuning": FINAL_TAG}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for a, (title, tag) in zip(ax, emb_tags.items()):
    path = RUNS / f"{tag}_emb_test.npy"
    if not path.exists():
        a.set_title(f"{title}: sin datos"); continue
    Z = np.load(path).astype(np.float32)
    Y2 = TSNE(n_components=2, perplexity=min(30, len(Z) // 4), random_state=0, init="pca").fit_transform(Z)
    for k, cat in enumerate(cat_names):
        a.scatter(*Y2[yc == k].T, s=8, alpha=0.8, label=cat)
    a.set_title(f"t-SNE: {title}"); a.set_xticks([]); a.set_yticks([])
ax[1].legend(fontsize=8, markerscale=2)
plt.tight_layout(); plt.savefig(FIG / "tsne_embeddings.png", dpi=150); plt.show()

## 12. Figuras y cifras para el informe

Las figuras del informe se exportan a `report/figs/` y las cifras citadas a `outputs/tables/resumen_informe.json`, para que el informe y el notebook coincidan.

In [ ]:
plt.rcParams.update({"font.size": 8})
order = np.argsort(pc.f1.values)
fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.6), gridspec_kw={"width_ratios": [1.6, 1]})
cols = {cat: plt.cm.tab10(k) for k, cat in enumerate(cat_names)}
ax[0].bar(range(len(CLASSES)), pc.f1.values[order], color=[cols[pc.categoria.values[i]] for i in order])
ax[0].set_xticks(range(len(CLASSES)), [CLASSES[i] for i in order], rotation=90, fontsize=4.5)
ax[0].set_ylabel("F1 (prueba)"); ax[0].set_ylim(max(0, pc.f1.min() - 0.1), 1.0); ax[0].set_title("(a) F1 por clase, ordenada")
ax[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=cols[c]) for c in cat_names], labels=cat_names, fontsize=5, loc="lower right")
ax[1].imshow(cmc / cmc.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(5):
    for j in range(5):
        ax[1].text(j, i, cmc[i, j], ha="center", va="center", fontsize=6, color="white" if cmc[i, j] / cmc[i].sum() > 0.5 else "black")
ab = ["Anim.", "Nat.", "Hum.", "Dom.", "Urb."]
ax[1].set_xticks(range(5), ab, fontsize=6); ax[1].set_yticks(range(5), ab, fontsize=6)
ax[1].set_xlabel("Predicción"); ax[1].set_ylabel("Real"); ax[1].set_title("(b) Confusión por categoría")
plt.tight_layout(); plt.savefig(REPORT_FIGS / "fig1_resultados.pdf", bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(3.4, 3.1))
sub = [CLASSES.index(c) for c in pd.unique(pd.concat([pares.head(6).real, pares.head(6).predicha]))]
ax.imshow(cm[np.ix_(sub, sub)], cmap="Blues")
for i in range(len(sub)):
    for j in range(len(sub)):
        ax.text(j, i, cm[sub[i], sub[j]], ha="center", va="center", fontsize=6, color="red" if i != j and cm[sub[i], sub[j]] else "black")
ax.set_xticks(range(len(sub)), [CLASSES[i] for i in sub], rotation=60, ha="right", fontsize=5.5)
ax.set_yticks(range(len(sub)), [CLASSES[i] for i in sub], fontsize=5.5); ax.set_title("Clases más confundidas", fontsize=8)
plt.tight_layout(); plt.savefig(REPORT_FIGS / "fig2_confusiones.pdf", bbox_inches="tight"); plt.show()
plt.rcParams.update({"font.size": 10})

resumen_informe = {
    "datos": {"clips": int(len(meta)), "particion": tabla.clips.to_dict(), "grabaciones_compartidas": cruces,
              "fraccion_relleno": round(frac_pad, 3)},
    "linea_base": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in BASE.items()},
    "experimento_1": e1[SHOW].round(4).to_dict(orient="records"), "experimento_2": e2[SHOW].round(4).to_dict(orient="records"),
    "configuracion_final": FINAL, "experimento_3_media_std": agg["media ± std"].to_dict(),
    "f1_macro_ic95": [round(float(v), 4) for v in F1_CI], "diferencia_mfcc_ic95": [round(float(v), 4) for v in DIFF_CI],
    "peores_clases": pc.sort_values("f1").head(5)[["categoria", "precision", "recall", "f1"]].round(3).to_dict(orient="index"),
    "por_categoria": por_cat.round(3).to_dict(orient="index"),
    "pares_mas_confundidos": pares.head(5).to_dict(orient="records"), "dispositivo": gpu, "modo_rapido": QUICK,
}
(TAB / "resumen_informe.json").write_text(json.dumps(resumen_informe, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
print(json.dumps(resumen_informe, indent=1, ensure_ascii=False, default=str)[:3000])

## 13. Resultados y discusión

*Esta sección se completa con las salidas de la ejecución completa.*

## 14. Referencias

[1] K. J. Piczak, "ESC: Dataset for environmental sound classification," in *Proc. 23rd ACM Int. Conf. Multimedia*, 2015, pp. 1015–1018.

[2] Y. Gong, Y.-A. Chung, and J. Glass, "AST: Audio spectrogram transformer," in *Proc. Interspeech*, 2021, pp. 571–575.

[3] A. Vaswani *et al.*, "Attention is all you need," in *Proc. Adv. Neural Inf. Process. Syst. (NeurIPS)*, 2017, pp. 5998–6008.

[4] A. Dosovitskiy *et al.*, "An image is worth 16x16 words: Transformers for image recognition at scale," in *Proc. Int. Conf. Learn. Represent. (ICLR)*, 2021.

[5] J. F. Gemmeke *et al.*, "Audio Set: An ontology and human-labeled dataset for audio events," in *Proc. IEEE Int. Conf. Acoust., Speech Signal Process. (ICASSP)*, 2017, pp. 776–780.

[6] H. Touvron *et al.*, "Training data-efficient image transformers & distillation through attention," in *Proc. Int. Conf. Mach. Learn. (ICML)*, 2021, pp. 10347–10357.

[7] I. Goodfellow, Y. Bengio, and A. Courville, *Deep Learning*. Cambridge, MA, USA: MIT Press, 2016.

[8] I. Loshchilov and F. Hutter, "Decoupled weight decay regularization," in *Proc. Int. Conf. Learn. Represent. (ICLR)*, 2019.

[9] D. S. Park *et al.*, "SpecAugment: A simple data augmentation method for automatic speech recognition," in *Proc. Interspeech*, 2019, pp. 2613–2617.

[10] S. Davis and P. Mermelstein, "Comparison of parametric representations for monosyllabic word recognition in continuously spoken sentences," *IEEE Trans. Acoust., Speech, Signal Process.*, vol. 28, no. 4, pp. 357–366, 1980.

[11] L. van der Maaten and G. Hinton, "Visualizing data using t-SNE," *J. Mach. Learn. Res.*, vol. 9, pp. 2579–2605, 2008.